# 08b -- XGBoost Hyperparameter Tuning (Scheme A vs. Scheme B)

Companion to `08_modeling.ipynb`. Section 8.6 compared Logistic Regression with Random Forest and XGBoost using **default**
hyperparameters, and Section 8.7 tuned **only** the winner (Logistic Regression). This notebook gives XGBoost a fair, tuned
comparison on both training schemes (Section 8.10), following the same leakage rules:

* fit on the **train** period, early-stop and select on **validation**; the **test** period stays locked until the very end;
* every selection step uses only information available before the test period (labels up to 2011-07-03);
* the final candidates are refit on train + validation exactly like Section 8.7 and evaluated **once** on test.

# Libs

In [1]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, spearmanr, uniform
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Helpers

Preprocessing and pooled metrics are identical to `08_modeling.ipynb`; weekly metrics follow Chapter 9 (rank each week's Scoring Population, cut the top 10%, average across weeks).

In [2]:
FULL_FEATURES = [
    "recency_days", "frequency_so_far", "monetary_so_far",
    "avg_order_value_so_far", "tenure_days", "mean_gap_so_far",
    "recency_to_typical_gap", "activity_last_90d",
    "distinct_products_so_far", "cancel_count_so_far",
    "cancel_value_share_so_far",
]
LOG_FEATURES = [
    "monetary_so_far", "avg_order_value_so_far", "tenure_days",
    "mean_gap_so_far", "recency_to_typical_gap", "activity_last_90d",
    "distinct_products_so_far", "cancel_count_so_far",
    "frequency_so_far",
]
HYPERPARAMS = [
    "learning_rate", "max_depth", "min_child_weight", "gamma",
    "subsample", "colsample_bytree", "reg_lambda", "reg_alpha",
]


def prepare_features(df):
    """Select the Full feature set and log1p the skewed columns (as in 08)."""
    X = df[FULL_FEATURES].copy()
    for col in LOG_FEATURES:
        X[col] = np.log1p(X[col])
    return X.replace([np.inf, -np.inf], np.nan)


def fit_preprocessing(train_df, impute=True):
    """Fit median imputer + scaler on train rows only; return a transform function.

    impute=False keeps NaN so XGBoost can learn its own default direction.
    """
    X_train = prepare_features(train_df)
    imputer = SimpleImputer(strategy="median").fit(X_train) if impute else None
    scaler = StandardScaler().fit(
        imputer.transform(X_train) if impute else X_train
    )

    def transform(df):
        X = prepare_features(df)
        return scaler.transform(imputer.transform(X) if impute else X)

    return transform


def top_k_labels(y_true, scores, k_frac=0.10):
    """Labels of the top k_frac rows by score."""
    k = max(int(round(len(scores) * k_frac)), 1)
    top_idx = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[top_idx]


def pooled_metrics(y_true, scores):
    """ROC-AUC, PR-AUC and top-10% metrics pooled over all rows (as in 08)."""
    y_true = np.asarray(y_true)
    top = top_k_labels(y_true, scores)
    return {
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "precision_at_10": top.mean(),
        "lift_at_10": top.mean() / y_true.mean(),
        "gain_at_10": top.sum() / y_true.sum(),
    }


def weekly_metrics(df, scores):
    """Precision/Lift/Gain at 10% computed per scoring week, then averaged (Ch. 9)."""
    t = df[["snapshot_date", "churn"]].assign(score=np.asarray(scores))
    rows = []
    for _, week in t.groupby("snapshot_date"):
        top = top_k_labels(week["churn"].values, week["score"].values)
        rows.append((
            top.mean(),
            top.mean() / week["churn"].mean(),
            top.sum() / week["churn"].sum(),
        ))
    rows = np.array(rows)
    return {
        "weekly_precision_at_10": rows[:, 0].mean(),
        "weekly_lift_at_10": rows[:, 1].mean(),
        "weekly_lift_at_10_std": rows[:, 1].std(),
        "weekly_gain_at_10": rows[:, 2].mean(),
    }


def apply_episode_cooldown(active_grid, cooldown_days):
    """Select one training snapshot per eligibility episode (same rule as 07/08)."""
    cooldown = pd.Timedelta(days=cooldown_days)
    ordered = active_grid.sort_values(["customer_id", "snapshot_date"])
    selected_index = []
    for _, rows in ordered.groupby("customer_id"):
        next_allowed = None
        for row in rows.itertuples():
            if next_allowed is None or row.snapshot_date >= next_allowed:
                selected_index.append(row.Index)
                next_allowed = row.snapshot_date + cooldown
    return active_grid.loc[selected_index]


def normalized_weights(customer_ids):
    """1 / rows-per-customer, rescaled to mean 1.

    Raw 1/n_i weights average ~0.06, which silently multiplies XGBoost's
    min_child_weight, gamma and reg_lambda (and Logistic Regression's
    regularization) by ~16x. Rescaling to mean 1 keeps every
    hyperparameter's meaning identical between Scheme A and Scheme B while
    preserving the relative weights (each customer still sums to the same
    total).
    """
    counts = customer_ids.map(customer_ids.value_counts())
    weights = 1.0 / counts.values
    return weights * len(weights) / weights.sum()

# Load

In [3]:
H = 90
COOLDOWN_DAYS = H

modeling_dataset = pd.read_parquet(
    "../data/processed/modeling_dataset_h90.parquet"
)
train_a = modeling_dataset[
    (modeling_dataset["split"] == "train")
    & modeling_dataset["in_training_sample"]
]
train_b = modeling_dataset[modeling_dataset["split"] == "train"]
val_df = modeling_dataset[modeling_dataset["split"] == "validation"]
test_df = modeling_dataset[modeling_dataset["split"] == "test"]
weights_b = normalized_weights(train_b["customer_id"])

print(f"Scheme A train rows: {len(train_a):,}")
print(f"Scheme B train rows: {len(train_b):,} (weights rescaled to mean {weights_b.mean():.2f})")
print(f"Validation rows: {len(val_df):,} | Test rows: {len(test_df):,} (locked)")

Scheme A train rows: 6,453
Scheme B train rows: 61,415 (weights rescaled to mean 1.00)
Validation rows: 21,514 | Test rows: 24,252 (locked)


## 1. Reproduce Section 8.6 (default hyperparameters)

Same models and settings as `08_modeling.ipynb`, plus the default XGBoost trained on Scheme B, to anchor the comparison.

In [4]:
def score_on(model, train_df, eval_df, sample_weight=None, impute=True):
    transform = fit_preprocessing(train_df, impute=impute)
    model.fit(transform(train_df), train_df["churn"], sample_weight=sample_weight)
    return model.predict_proba(transform(eval_df))[:, 1]


def default_xgb():
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        eval_metric="logloss", random_state=0, n_jobs=-1,
    )


reference_models = {
    "LR C=0.01 | A (Section 8.7 final)": (
        LogisticRegression(C=0.01, max_iter=1000, solver="liblinear"), train_a, None),
    "XGBoost default | A (Section 8.6)": (default_xgb(), train_a, None),
    "XGBoost default | B": (default_xgb(), train_b, weights_b),
}
reference_rows = []
for name, (model, train_df, weights) in reference_models.items():
    scores = score_on(model, train_df, val_df, weights)
    reference_rows.append({"model": name, **pooled_metrics(val_df["churn"], scores),
                           **weekly_metrics(val_df, scores)})
reference_results = pd.DataFrame(reference_rows).set_index("model")
reference_results.round(3)

,roc_auc,pr_auc,precision_at_10,lift_at_10,gain_at_10,weekly_precision_at_10,weekly_lift_at_10,weekly_lift_at_10_std,weekly_gain_at_10
model,,,,,,,,,
LR C=0.01 | A (Section 8.7 final),0.772,0.750,0.799,1.512,0.151,0.795,1.507,0.016,0.151
XGBoost default | A (Section 8.6),0.717,0.703,0.770,1.457,0.146,0.771,1.464,0.041,0.146
XGBoost default | B,0.759,0.742,0.810,1.533,0.153,0.810,1.538,0.049,0.154


## 2. Search space

| Group | Hyperparameter | Role | Search range |
|---|---|---|---|
| Boosting | `n_estimators` | number of trees | early stopping on validation PR-AUC (patience 100, max 3,000) |
| Boosting | `learning_rate` ($\eta$) | shrinkage of each tree | 0.01 - 0.3, log-uniform |
| Tree complexity | `max_depth` | interaction order a tree can express | 2 - 8 |
| Tree complexity | `min_child_weight` | minimum hessian sum per leaf; for log-loss $h=p(1-p)\le0.25$ | 1 - 200, log-uniform |
| Tree complexity | `gamma` | minimum loss reduction to make a split | 0 - 5 |
| Randomness | `subsample` | row fraction per tree | 0.5 - 1 |
| Randomness | `colsample_bytree` | feature fraction per tree | 0.5 - 1 |
| Leaf regularization | `reg_lambda` ($\lambda$) | L2 penalty on leaf weights, $w^*=-G/(H+\lambda)$ | 0.1 - 100, log-uniform |
| Leaf regularization | `reg_alpha` ($\alpha$) | L1 penalty on leaf weights | 0.001 - 10, log-uniform |

Not tuned on purpose: `scale_pos_weight` / `max_delta_step` (churn is ~47%, not a rare class, and reweighting classes would
distort the probabilities that Chapter 9 uses as values), `colsample_bylevel/bynode`, `max_leaves/grow_policy` (redundant with
the parameters above), and `tree_method="hist"`, `max_bin`, `objective` (technical settings).

**Why random search.** Only a few hyperparameters usually matter; for the same budget random search tries many more distinct
values of each one than a grid (Bergstra & Bengio, 2012). Scale-type parameters are sampled on a log scale. The **same 120
configurations** are run on Scheme A and Scheme B, so the two schemes are compared pairwise.

In [5]:
SEARCH_SPACE = {
    "learning_rate": loguniform(0.01, 0.3),
    "max_depth": randint(2, 9),
    "min_child_weight": loguniform(1, 200),
    "gamma": uniform(0, 5),
    "subsample": uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_lambda": loguniform(0.1, 100),
    "reg_alpha": loguniform(1e-3, 10),
}
N_ITER = 120
configs = list(ParameterSampler(SEARCH_SPACE, n_iter=N_ITER, random_state=42))

search_rows = []
for scheme, train_df, weights in [("A", train_a, None), ("B", train_b, weights_b)]:
    transform = fit_preprocessing(train_df)
    X_train, X_val = transform(train_df), transform(val_df)
    start = time.time()
    for cfg_id, params in enumerate(configs):
        model = XGBClassifier(
            n_estimators=3000, early_stopping_rounds=100, eval_metric="aucpr",
            tree_method="hist", random_state=0, n_jobs=-1, **params,
        )
        model.fit(X_train, train_df["churn"], sample_weight=weights,
                  eval_set=[(X_val, val_df["churn"])], verbose=False)
        scores = model.predict_proba(
            X_val, iteration_range=(0, model.best_iteration + 1)
        )[:, 1]
        search_rows.append({
            "scheme": scheme, "cfg": cfg_id,
            **{k: (int(v) if k == "max_depth" else float(v)) for k, v in params.items()},
            "n_estimators": model.best_iteration + 1,
            **pooled_metrics(val_df["churn"], scores),
            **weekly_metrics(val_df, scores),
        })
    print(f"Scheme {scheme}: {N_ITER} configs in {time.time() - start:.0f}s")
search = pd.DataFrame(search_rows)

Scheme A: 120 configs in 86s


Scheme B: 120 configs in 109s


In [6]:
summary = search.groupby("scheme")["pr_auc"].describe()[["min", "50%", "max"]]
summary["top10_spread"] = search.groupby("scheme")["pr_auc"].apply(
    lambda s: s.nlargest(10).max() - s.nlargest(10).min()
)
summary.round(4)

,min,50%,max,top10_spread
scheme,,,,
A,0.7331,0.7423,0.7498,0.0037
B,0.7487,0.7641,0.7673,0.0012


In [7]:
for scheme in ["A", "B"]:
    print(f"Top 5 configurations, Scheme {scheme} (validation)")
    display(
        search[search["scheme"] == scheme]
        .nlargest(5, "pr_auc")[["cfg"] + HYPERPARAMS
                               + ["n_estimators", "roc_auc", "pr_auc", "weekly_lift_at_10"]]
        .round(4)
    )

Top 5 configurations, Scheme A (validation)


,cfg,learning_rate,max_depth,min_child_weight,gamma,subsample,colsample_bytree,reg_lambda,reg_alpha,n_estimators,roc_auc,pr_auc,weekly_lift_at_10
69,69,0.0693,2,11.9806,4.1458,0.5282,0.6965,0.6610,0.0456,65,0.7718,0.7498,1.4974
48,48,0.0841,2,162.8277,4.1032,0.5347,0.9056,0.3867,4.1822,60,0.7689,0.7470,1.4889
100,100,0.0168,2,42.3432,3.9430,0.6023,0.9469,0.2688,1.3010,200,0.7690,0.7466,1.4669
44,44,0.0117,2,5.2068,0.5804,0.5086,0.5072,0.3357,8.2802,306,0.7718,0.7466,1.4744
118,118,0.0301,3,1.0713,2.0320,0.5133,0.5705,0.4204,0.0022,69,0.7708,0.7465,1.4782


Top 5 configurations, Scheme B (validation)


,cfg,learning_rate,max_depth,min_child_weight,gamma,subsample,colsample_bytree,reg_lambda,reg_alpha,n_estimators,roc_auc,pr_auc,weekly_lift_at_10
137,17,0.0414,6,39.2027,2.5537,0.5841,0.5035,0.5400,0.0120,14,0.7804,0.7673,1.5961
208,88,0.0189,7,71.5022,0.8034,0.5258,0.6230,33.6467,0.0017,80,0.7784,0.7672,1.5744
235,115,0.0402,8,178.8754,3.2200,0.7040,0.6354,39.9008,2.2864,58,0.7780,0.7667,1.5706
203,83,0.1132,8,87.7169,1.2622,0.9336,0.5183,0.1234,0.9965,5,0.7785,0.7665,1.6063
138,18,0.0395,7,172.2264,2.7905,0.7486,0.6094,0.5693,7.0760,10,0.7777,0.7665,1.5905


## 3. Sensitivity: which hyperparameters matter

Spearman correlation between each hyperparameter and validation PR-AUC across the 120 configurations (negative = smaller values are better).

In [8]:
sensitivity = pd.DataFrame({
    scheme: {
        hp: spearmanr(search.loc[search["scheme"] == scheme, hp],
                      search.loc[search["scheme"] == scheme, "pr_auc"])[0]
        for hp in HYPERPARAMS
    }
    for scheme in ["A", "B"]
})
sensitivity.round(3)

,A,B
learning_rate,-0.027,-0.072
max_depth,-0.444,-0.411
min_child_weight,-0.175,0.275
gamma,0.185,-0.080
subsample,-0.084,-0.129
colsample_bytree,-0.441,-0.607
reg_lambda,-0.119,0.123
reg_alpha,0.047,0.080


In [9]:
binned = []
for hp, bins in [("max_depth", [1.5, 3.5, 5.5, 8.5]),
                 ("min_child_weight", [1, 10, 50, 200]),
                 ("learning_rate", [0.01, 0.03, 0.1, 0.3]),
                 ("colsample_bytree", [0.5, 0.75, 1.0])]:
    t = (search.assign(bin=pd.cut(search[hp], bins, include_lowest=True))
         .groupby(["scheme", "bin"], observed=True)["pr_auc"].median()
         .unstack(0))
    t.index = [f"{hp} {b}" for b in t.index]
    binned.append(t)
pd.concat(binned).round(4)

scheme,A,B
"max_depth (1.499, 3.5]",0.7441,0.7649
"max_depth (3.5, 5.5]",0.7416,0.7637
"max_depth (5.5, 8.5]",0.7412,0.7620
"min_child_weight (0.999, 10.0]",0.7431,0.7638
"min_child_weight (10.0, 50.0]",0.7430,0.7638
"min_child_weight (50.0, 200.0]",0.7412,0.7651
"learning_rate (0.009000000000000001, 0.03]",0.7418,0.7642
"learning_rate (0.03, 0.1]",0.7432,0.7644
"learning_rate (0.1, 0.3]",0.7415,0.7638
"colsample_bytree (0.499, 0.75]",0.7432,0.7651


## 4. Robustness on a second pre-test season

Validation covers Feb-Apr 2011 only. To avoid picking a configuration that just fits that season, the top 5 configurations per
scheme are re-trained (fixed `n_estimators`) on all labelable snapshots up to 90 days before **2010-12-06** and evaluated on the
8 weeks starting 2010-12-06 (Dec-Jan, peak churn season). The final choice is the best **mean** PR-AUC over validation and this
season. Both windows end before the test period, so the test set is still untouched.

In [10]:
labelable = modeling_dataset[modeling_dataset["labelable"]]
season_start = pd.Timestamp("2010-12-06")
season_eval = labelable[(labelable["snapshot_date"] >= season_start)
                        & (labelable["snapshot_date"] < season_start + pd.Timedelta(weeks=8))]
season_pool = labelable[labelable["snapshot_date"] <= season_start - pd.Timedelta(days=H)]
season_train = {"A": (apply_episode_cooldown(season_pool, COOLDOWN_DAYS), None),
                "B": (season_pool, normalized_weights(season_pool["customer_id"]))}


def xgb_from(row, n_estimators=None, **extra):
    params = {hp: (int(row[hp]) if hp == "max_depth" else float(row[hp])) for hp in HYPERPARAMS}
    return XGBClassifier(n_estimators=int(n_estimators or row["n_estimators"]),
                         tree_method="hist", random_state=0, n_jobs=-1, **params, **extra)


robust_rows = []
for scheme in ["A", "B"]:
    for _, row in search[search["scheme"] == scheme].nlargest(5, "pr_auc").iterrows():
        train_df, weights = season_train[scheme]
        scores = score_on(xgb_from(row), train_df, season_eval, weights)
        robust_rows.append({"scheme": scheme, "cfg": int(row["cfg"]),
                            "n_estimators": int(row["n_estimators"]),
                            "val_pr_auc": row["pr_auc"],
                            "season_pr_auc": average_precision_score(season_eval["churn"], scores)})
robust = pd.DataFrame(robust_rows)
robust["mean_pr_auc"] = robust[["val_pr_auc", "season_pr_auc"]].mean(axis=1)

lr_season = score_on(LogisticRegression(C=0.01, max_iter=1000, solver="liblinear"),
                     season_train["A"][0], season_eval)
print(f"Reference LR C=0.01 | A on the Dec-Jan season: PR-AUC "
      f"{average_precision_score(season_eval['churn'], lr_season):.4f}")
robust.round(4)

/var/folders/9c/1yz9h2n56c98554xgyfj6_dr0000gn/T/ipykernel_7943/1810224600.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  & (labelable["snapshot_date"] < season_start + pd.Timedelta(weeks=8))]
/var/folders/9c/1yz9h2n56c98554xgyfj6_dr0000gn/T/ipykernel_7943/1810224600.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  season_pool = labelable[labelable["snapshot_date"] <= season_start - pd.Timedelta(days=H)]
/var/folders/9c/1yz9h2n56c98554xgyfj6_dr0000gn/T/ipykernel_7943/4150270203.py:88: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integer

Reference LR C=0.01 | A on the Dec-Jan season: PR-AUC 0.8147


,scheme,cfg,n_estimators,val_pr_auc,season_pr_auc,mean_pr_auc
0,A,69,65,0.7498,0.8155,0.7826
1,A,48,60,0.7470,0.8050,0.7760
2,A,100,200,0.7466,0.8115,0.7791
3,A,44,306,0.7466,0.8116,0.7791
4,A,118,69,0.7465,0.8160,0.7813
5,B,17,14,0.7673,0.8135,0.7904
6,B,88,80,0.7672,0.8143,0.7908
7,B,115,58,0.7667,0.8149,0.7908
8,B,83,5,0.7665,0.8098,0.7881
9,B,18,10,0.7665,0.8108,0.7886


In [11]:
chosen = {}
for scheme in ["A", "B"]:
    cfg = robust[robust["scheme"] == scheme].sort_values("mean_pr_auc", ascending=False).iloc[0]["cfg"]
    chosen[scheme] = search[(search["scheme"] == scheme) & (search["cfg"] == cfg)].iloc[0]
pd.DataFrame({s: chosen[s][HYPERPARAMS + ["n_estimators", "pr_auc"]] for s in chosen}).round(4)

,A,B
learning_rate,0.069279,0.040156
max_depth,2,8
min_child_weight,11.980557,178.875391
gamma,4.145821,3.219998
subsample,0.528188,0.703992
colsample_bytree,0.696495,0.635372
reg_lambda,0.66103,39.90077
reg_alpha,0.045633,2.286417
n_estimators,65,58
pr_auc,0.74978,0.766686


## 5. Variants of the chosen configuration

Native missing-value handling (no median imputation) and monotone constraints from business logic: churn risk can only rise with recency and recency-to-typical-gap, and only fall with frequency, monetary value and recent activity.

In [12]:
MONOTONE = "(1,-1,-1,0,0,0,1,-1,0,0,0)"  # order = FULL_FEATURES
variant_rows = []
for scheme, train_df, weights in [("A", train_a, None), ("B", train_b, weights_b)]:
    for variant, extra, impute in [("chosen (median imputation)", {}, True),
                                   ("native NaN handling", {}, False),
                                   ("monotone constraints", {"monotone_constraints": MONOTONE}, True)]:
        scores = score_on(xgb_from(chosen[scheme], **extra), train_df, val_df, weights, impute)
        variant_rows.append({"scheme": scheme, "variant": variant,
                             **pooled_metrics(val_df["churn"], scores),
                             **weekly_metrics(val_df, scores)})
pd.DataFrame(variant_rows).set_index(["scheme", "variant"])[
    ["roc_auc", "pr_auc", "weekly_lift_at_10"]].round(4)

roc_auc  pr_auc  weekly_lift_at_10
scheme variant                                                       
A      chosen (median imputation)   0.7718  0.7498             1.4974
       native NaN handling          0.7722  0.7503             1.4966
       monotone constraints         0.7730  0.7471             1.4967
B      chosen (median imputation)   0.7780  0.7667             1.5706
       native NaN handling          0.7778  0.7672             1.5801
       monotone constraints         0.7783  0.7666             1.5765

## 6. Logistic Regression on Scheme B (same weight normalization)

For a like-for-like comparison, Logistic Regression is also tuned on Scheme B with the rescaled weights.

In [13]:
lr_b_rows = []
for c in [0.001, 0.01, 0.1, 1.0, 10.0]:
    scores = score_on(LogisticRegression(C=c, max_iter=2000, solver="liblinear"),
                      train_b, val_df, weights_b)
    lr_b_rows.append({"C": c, **pooled_metrics(val_df["churn"], scores)})
lr_b = pd.DataFrame(lr_b_rows).set_index("C")
best_c_b = lr_b["pr_auc"].round(4).idxmax()
print(f"Best C for LR | B (validation PR-AUC, ties -> stronger regularization): {best_c_b}")
lr_b.round(4)

Best C for LR | B (validation PR-AUC, ties -> stronger regularization): 1.0


,roc_auc,pr_auc,precision_at_10,lift_at_10,gain_at_10
C,,,,,
0.001,0.7801,0.7689,0.8489,1.6059,0.1606
0.010,0.7808,0.7696,0.8452,1.5988,0.1599
0.100,0.7809,0.7698,0.8447,1.5979,0.1598
1.000,0.7810,0.7699,0.8447,1.5979,0.1598
10.000,0.7810,0.7699,0.8452,1.5988,0.1599


## 7. Final refit on train + validation and a single test evaluation

Same recipe as Section 8.7: the Training Population is re-derived over train + validation (Scheme A: episode rule; Scheme B: all
rows with weights recomputed on that period). `n_estimators` is fixed to the early-stopping value found above. Differences against
the team's final model are tested with a **customer-level bootstrap** (500 resamples of customers), because rows of the same
customer in consecutive weeks are not independent.

In [14]:
refit_pool = modeling_dataset[
    (modeling_dataset["snapshot_date"] >= train_a["snapshot_date"].min())
    & (modeling_dataset["snapshot_date"] <= val_df["snapshot_date"].max())
    & modeling_dataset["labelable"]
]
refit_a = apply_episode_cooldown(refit_pool, COOLDOWN_DAYS)
refit_b = refit_pool
refit_weights_b = normalized_weights(refit_b["customer_id"])
print(f"Refit rows: Scheme A {len(refit_a):,} | Scheme B {len(refit_b):,}")

final_candidates = {
    "LR C=0.01 | A (team final)": (
        LogisticRegression(C=0.01, max_iter=1000, solver="liblinear"), refit_a, None),
    f"LR C={best_c_b:g} | B": (
        LogisticRegression(C=best_c_b, max_iter=2000, solver="liblinear"), refit_b, refit_weights_b),
    "XGBoost default | A (Section 8.6)": (default_xgb(), refit_a, None),
    "XGBoost tuned | A": (xgb_from(chosen["A"]), refit_a, None),
    "XGBoost tuned | B": (xgb_from(chosen["B"]), refit_b, refit_weights_b),
}
test_scores, test_rows = {}, []
for name, (model, train_df, weights) in final_candidates.items():
    test_scores[name] = score_on(model, train_df, test_df, weights)
    test_rows.append({"model": name, "train_rows": len(train_df),
                      **pooled_metrics(test_df["churn"], test_scores[name]),
                      **weekly_metrics(test_df, test_scores[name])})
test_results = pd.DataFrame(test_rows).set_index("model")
test_results.round(4)

/var/folders/9c/1yz9h2n56c98554xgyfj6_dr0000gn/T/ipykernel_7943/4150270203.py:88: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  cooldown = pd.Timedelta(days=cooldown_days)


Refit rows: Scheme A 12,019 | Scheme B 120,376


,train_rows,roc_auc,pr_auc,precision_at_10,lift_at_10,gain_at_10,weekly_precision_at_10,weekly_lift_at_10,weekly_lift_at_10_std,weekly_gain_at_10
model,,,,,,,,,,
LR C=0.01 | A (team final),12019,0.7645,0.6575,0.7575,1.8912,0.1891,0.7601,1.9022,0.0523,0.1904
LR C=1 | B,120376,0.7643,0.6606,0.7720,1.9273,0.1927,0.7635,1.9117,0.0589,0.1913
XGBoost default | A (Section 8.6),12019,0.7471,0.6330,0.7299,1.8223,0.1822,0.7307,1.8328,0.0946,0.1834
XGBoost tuned | A,12019,0.7657,0.6517,0.7518,1.8768,0.1877,0.7455,1.8691,0.0840,0.1871
XGBoost tuned | B,120376,0.7709,0.6628,0.7579,1.8923,0.1892,0.7524,1.8844,0.0628,0.1886


In [15]:
baseline = "LR C=0.01 | A (team final)"
rng = np.random.default_rng(0)
customers = test_df["customer_id"].values
y_test = test_df["churn"].values
unique_customers = np.unique(customers)
rows_of = {c: np.where(customers == c)[0] for c in unique_customers}

bootstrap_rows = []
for name in [n for n in test_scores if n != baseline]:
    d_pr, d_roc = [], []
    for _ in range(500):
        idx = np.concatenate([rows_of[c] for c in
                              rng.choice(unique_customers, len(unique_customers), replace=True)])
        d_pr.append(average_precision_score(y_test[idx], test_scores[name][idx])
                    - average_precision_score(y_test[idx], test_scores[baseline][idx]))
        d_roc.append(roc_auc_score(y_test[idx], test_scores[name][idx])
                     - roc_auc_score(y_test[idx], test_scores[baseline][idx]))
    bootstrap_rows.append({
        "model": name,
        "delta_pr_auc": np.mean(d_pr),
        "pr_auc_ci_low": np.percentile(d_pr, 2.5), "pr_auc_ci_high": np.percentile(d_pr, 97.5),
        "delta_roc_auc": np.mean(d_roc),
        "roc_auc_ci_low": np.percentile(d_roc, 2.5), "roc_auc_ci_high": np.percentile(d_roc, 97.5),
    })
pd.DataFrame(bootstrap_rows).set_index("model").round(4)

,delta_pr_auc,pr_auc_ci_low,pr_auc_ci_high,delta_roc_auc,roc_auc_ci_low,roc_auc_ci_high
model,,,,,,
LR C=1 | B,0.0027,-0.0035,0.0095,-0.0003,-0.0024,0.0021
XGBoost default | A (Section 8.6),-0.0249,-0.0407,-0.0080,-0.0180,-0.0278,-0.0087
XGBoost tuned | A,-0.0058,-0.0132,0.0022,0.0011,-0.0027,0.0049
XGBoost tuned | B,0.0055,-0.0041,0.0142,0.0064,0.0025,0.0105


In [16]:
json.dump({s: {**{hp: float(chosen[s][hp]) for hp in HYPERPARAMS},
               "n_estimators": int(chosen[s]["n_estimators"])} for s in chosen},
          open("../data/processed/xgboost_tuned_params.json", "w"), indent=1)
print("Saved chosen hyperparameters to ../data/processed/xgboost_tuned_params.json")

Saved chosen hyperparameters to ../data/processed/xgboost_tuned_params.json


## 8. Conclusion

**Default vs. tuned.** The default XGBoost of Section 8.6 (300 trees, depth 4, no regularization, no early stopping) overfits the 6,453 episode rows: validation PR-AUC 0.703, and on test it is significantly worse than the final Logistic Regression (ΔPR-AUC −0.025, 95% CI [−0.041, −0.008]). Tuning closes that gap: validation PR-AUC 0.750 (Scheme A) and 0.767 (Scheme B).

**Tuned XGBoost vs. Logistic Regression on test** (refit on train + validation, evaluated once, customer-level bootstrap vs. the team's final LR):

| Model | ROC-AUC | PR-AUC | Weekly Lift@10% | ΔPR-AUC [95% CI] | ΔROC-AUC [95% CI] |
|---|---|---|---|---|---|
| LR C=0.01, A (team final) | 0.765 | 0.658 | 1.90 | — | — |
| LR C=1, B | 0.764 | 0.661 | 1.91 | +0.003 [−0.004, +0.010] | 0.000 [−0.002, +0.002] |
| XGBoost default, A | 0.747 | 0.633 | 1.83 | −0.025 [−0.041, −0.008] | −0.018 [−0.028, −0.009] |
| XGBoost tuned, A | 0.766 | 0.652 | 1.87 | −0.006 [−0.013, +0.002] | +0.001 [−0.003, +0.005] |
| XGBoost tuned, B | 0.771 | 0.663 | 1.88 | +0.006 [−0.004, +0.014] | +0.006 [+0.003, +0.011] |

Tuned XGBoost on Scheme B has a small but statistically significant edge in ROC-AUC (+0.006), no significant edge in PR-AUC, and a slightly *lower* weekly Lift@10% — the metric that drives weekly targeting. **Logistic Regression remains the final model**: equal operational performance, directly interpretable coefficients, and already used by the scoring app.

**What tuning reveals about the data.** The search surface is flat (top-10 configurations within 0.004 PR-AUC on A and 0.001 on B). The hyperparameters that matter are the ones controlling tree complexity: shallower trees (`max_depth`, ρ ≈ −0.44 on A, −0.41 on B) and fewer features per tree (`colsample_bytree`, ρ ≈ −0.44 / −0.61) are better, and on Scheme B larger `min_child_weight` helps (ρ ≈ +0.28). `learning_rate` barely matters (ρ ≈ 0) because early stopping compensates with the number of trees. The chosen configurations are heavily constrained (A: depth 2, 65 trees; B: depth 8 but `min_child_weight` 179 and `reg_lambda` 40, 58 trees) — the signal in these 11 features is close to monotone and additive, which is exactly where a regularized linear model is hard to beat. Native NaN handling and monotone constraints change validation PR-AUC by less than 0.003.

**Scheme A vs. B.** With weights rescaled to mean 1, Scheme B is better on validation for both algorithms (+0.02 PR-AUC), but after the train + validation refit the difference on test is not significant for Logistic Regression (ΔPR-AUC +0.003, CI includes 0). The episode scheme (A) remains an acceptable default at this data size.